# VITS → B1 droid — Colab training (GAN fine-tune)

Fine-tunes VITS on the `Dmi1tr13/ljspeech-b1` droid dataset (branch `finetune`).

This is a **full GAN fine-tune**: our `vits_finetune` package now trains the decoder
against a from-scratch HiFi-GAN discriminator (multi-period + multi-scale) with
adversarial + feature-matching losses — the part our earlier mel-only loop was
missing (it produced noise, WER 90–120%). The training log shows both
`loss_d` (discriminator) and `loss_g` (generator).

**Before running:** `Runtime → Change runtime type → GPU (T4/L4) → Save`.

All logic lives in the source code (held-out test set, quiet logs,
`--max-train-clips`). This notebook just clones, installs, trains, and listens.

## 1. Clone the repo (branch with the fine-tuning fixes)

In [ ]:
!git clone -b finetune https://github.com/Dmitri1S9/DL.git project
%cd project

## 2. Install dependencies
System `espeak-ng` (the VITS tokenizer needs it) + Python deps. We skip the `torch+cu128` pin (a local GPU build unavailable on Colab) and use Colab's preinstalled torch.

In [ ]:
!apt-get -qq install -y espeak-ng
!grep -vE '^(torch|pedalboard)' requirements.txt > requirements-colab.txt
!pip install -q -r requirements-colab.txt
print('OK')

## 3. Sanity check — GPU + espeak

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
!espeak-ng --version

## 4. Train (GAN)
Each step now does two updates: a **discriminator** step then a **generator** step
(watch `loss_d` / `loss_g` in the log). batch 2, default LR (2e-4), ~2000 steps.
`--max-train-clips 4000` ÷ batch 2 = 2000 steps in one epoch; `--num-workers 4` splits the espeak
memory-map load across processes (Colab can't raise `vm.max_map_count`). The last 500 clips stay held
out for evaluation regardless. Adversarial training is slower per step than the old mel-only loop —
expect noticeably longer epochs, but real audio quality in return.

In [ ]:
!PYTHONPATH=src python -m vits_finetune.train \
    --batch-size 2 --num-epochs 1 --max-train-clips 4000 --num-workers 4

## 5. Listen — pretrained vs fine-tuned

In [ ]:
TEXT = 'Roger roger. All units, proceed with the mission. Standing by.'
!PYTHONPATH=src python -m vits_finetune.synthesize --text "$TEXT" --out demo_pretrained.wav
!PYTHONPATH=src python -m vits_finetune.synthesize --checkpoint models/vits_finetune/epoch_1.pt --text "$TEXT" --out demo_finetuned.wav

from IPython.display import Audio, display
print('=== Pretrained (natural) ==='); display(Audio('demo_pretrained.wav'))
print('=== Fine-tuned (droid) ===');   display(Audio('demo_finetuned.wav'))

## 6. Evaluate intelligibility with Whisper (our `src/evaluation` module)
We call our own evaluation code (`evaluation.metrics.compute_wer` / `compute_cer`) rather than
re-implementing it inline — same Whisper + jiwer method, but the one shipped in `src/`.
Low WER = intelligible, ~100% = garbage.

In [ ]:
import sys
sys.path.insert(0, 'src')  # import root is src/ (see README)
from evaluation.metrics import compute_cer, compute_wer

ref = [TEXT.lower().strip('.').strip()]
for tag, wav in [('pretrained', 'demo_pretrained.wav'), ('finetuned', 'demo_finetuned.wav')]:
    print(f'=== {tag} ===')
    compute_wer([wav], ref)
    compute_cer([wav], ref)

## 7. Full evaluation on the held-out test set (for the report)
The single-sentence check above is noisy. This scores **many held-out LJSpeech clips**
(never seen in training) for a stable WER/CER, pretrained vs fine-tuned, via our
`evaluation.evaluate` module. Downloads LJSpeech once (~2.6 GB) to build references.

- **WER / CER** — Whisper transcribes our synth and compares to the prompt text → intelligibility.
- **MCD** — distance to the *real LJSpeech (female) recording*; for the droid model this is
  expected to be high (different timbre), so read it as a sanity number, not voice quality.

Bump `--limit` / `LIMIT` to 500 for the final report number; 100 is a fast, stable estimate.

In [ ]:
import sys
sys.path.insert(0, 'src')
from pathlib import Path

from core import config
from data.prepare import prepare
from evaluation.evaluate import compare, evaluate
from model.synthesize import generate_for_manifest as gen_pretrained
from vits_finetune.synthesize import generate_for_manifest as gen_finetuned

LIMIT = 100                                  # held-out clips to score (500 = full test set)
CKPT = 'models/vits_finetune/epoch_1.pt'     # adjust to your latest checkpoint

# 1. Build the held-out test set (manifest + real LJSpeech references). No leakage:
#    these are the held-out clips, never used in training.
prepare(limit=LIMIT)

# 2. Synthesize the same prompts with both checkpoints.
gen_pre = Path('audio/generated_pretrained')
gen_ft = Path('audio/generated_finetuned')
gen_pretrained(config.TEST_MANIFEST, gen_pre, checkpoint=config.PRETRAINED)
gen_finetuned(CKPT, config.TEST_MANIFEST, gen_ft)

# 3. Score both and print the before/after table.
res_pre = evaluate(gen_pre, config.TEST_MANIFEST, label='pretrained')
res_ft = evaluate(gen_ft, config.TEST_MANIFEST, label='finetuned')
compare([res_pre, res_ft])